# Resource Estimation for Simulating a 2D Ising Model Hamiltonian

In this notebook we demonstrate how to estimate the resources for quantum
dynamics, specifically the simulation of an Ising model Hamiltonian on an
$N \times N$ 2D lattice using a *fourth-order Trotter–Suzuki product formula*
assuming a 2D qubit architecture with nearest-neighbor connectivity.

We use **qdk-chemistry** for domain-specific functionality (model
Hamiltonians, Trotter decomposition, circuit construction) and
**qsharp.qre** exclusively for quantum resource estimation.

> **Note:** This notebook requires a QRE-enabled build of qsharp
> (``qsharp.qre``). The qdk-chemistry cells (1–4) run independently.

## Background: 2D Ising Model

The Ising model is a mathematical model of ferromagnetism in a lattice
(in our case a 2D square lattice) with two kinds of terms in the
Hamiltonian: (i) an interaction term between adjacent sites and (ii) an
external magnetic field acting at each site.

$$
H = \underbrace{-J \sum_{\langle i, j \rangle} Z_i Z_j}_{B} - \underbrace{h \sum_j X_j}_{A}
$$

where $J$ is the interaction strength and $h$ is the external field strength.

## Creating the Ising Model

We use `qdk_chemistry` to define the lattice geometry and build the
Hamiltonian.  `LatticeGraph.square` creates a 2D square lattice and
`create_ising_hamiltonian` produces the corresponding `QubitHamiltonian`.

In [ ]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

lattice = LatticeGraph.square(10, 10, t=1.0)
hamiltonian = create_ising_hamiltonian(lattice, j=1.0, h=0.5)

print(f"Ising model on {lattice.num_sites}-site lattice "
      f"with {lattice.num_edges} edges")
print(f"QubitHamiltonian: {len(hamiltonian.pauli_strings)} Pauli terms, "
      f"{hamiltonian.num_qubits} qubits")

## Trotter Expansion

The time evolution $e^{-iHt}$ is simulated with the fourth-order product
formula. The `Trotter` builder implements arbitrary-order Suzuki
decompositions. With `optimize_term_ordering=True`, commuting terms are
grouped together for a more efficient decomposition.

In [ ]:
import math
from qdk_chemistry.algorithms.time_evolution.builder.trotter import Trotter

t = 1.0   # total simulation time
dt = 0.5  # time per Trotter step

trotter = Trotter(order=4, num_divisions=math.ceil(t / dt), optimize_term_ordering=True)
evolution = trotter.run(hamiltonian, time=t)

container = evolution.get_container()
print(f"4th-order Trotter: {len(container.step_terms)} Pauli "
      f"exponentials per step, {container.step_reps} repetitions")
print(f"Qubits: {evolution.get_num_qubits()}")

## Building the Circuit

The `TimeEvolutionUnitary` is converted to a `Circuit` using the
`PauliSequenceMapper`, which compiles the Pauli product formula into a
Q# circuit with QIR representation.

In [ ]:
from qdk_chemistry.algorithms.time_evolution.circuit_mapper import PauliSequenceMapper

mapper = PauliSequenceMapper()
circuit = mapper.run(evolution)

print(f"Circuit has QIR: {circuit.qir is not None}")
print(f"Circuit has Q#:  {circuit.qsharp is not None}")

## QRE Application

To run the full QRE resource estimation pipeline (with Majorana
architectures, lattice surgery, magic state factories, etc.), we wrap
our Q# operation in a `QSharpApplication`. This produces a trace that
QRE can analyze.

In [ ]:
from qsharp.qre.application import QSharpApplication

app = QSharpApplication(entry_expr=circuit._qsharp_op)
app_query = app.q()

app

## Majorana Architecture

A QRE `Architecture` is a container for an Instruction Set Architecture
(ISA). The `Majorana` subclass provides the ISA for a user-specified
measurement error rate.

In [ ]:
from qsharp.qre.models import Majorana

arch = Majorana(error_rate=1e-5)
arch

## Creating Application and ISA Queries

The trace query applies layouts (`ISATransform` instances) to convert the
operations from the application into logical operations supported by error
correction codes and magic state factories. The ISA query provides the
specific options for the code and magic state distillation.

In [ ]:
from qsharp.qre import estimate, PSSPC, LatticeSurgery
from qsharp.qre.models import ThreeAux, RoundBasedFactory

trace_query = (
    app_query
    * PSSPC.q(num_ts_per_rotation=[16, 17, 18, 19])
    * LatticeSurgery.q()
)

isa_query = (
    ThreeAux.q(distance=[11, 13, 15, 17, 19])
    * RoundBasedFactory.q(
        code_query=ThreeAux.q(distance=[5, 7, 11, 13, 15, 17, 19])
    )
)

results = estimate(
    app, arch, isa_query, trace_query,
    max_error=0.01,
    name="Majorana e-5, 3-aux",
)

## Visualizing and Understanding the Results

### Result Summary Table

QRE reports configurations that are Pareto-optimal in time and space
(and satisfy the specified error rate). We add columns for code distance,
compute qubits, T-gate budget, and factory layout.

In [ ]:
from qsharp.qre.instruction_ids import LATTICE_SURGERY
from qsharp.qre.property_keys import (
    NUM_TS_PER_ROTATION, DISTANCE, PHYSICAL_COMPUTE_QUBITS,
)

results.add_column(
    "compute_distance",
    lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE],
)
results.add_column(
    "compute qubits",
    lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS],
)
results.add_column(
    "num_ts_per_rotation",
    lambda entry: entry.properties[NUM_TS_PER_ROTATION],
)
results.add_factory_summary_column()
results.add_column(
    "cycle_time",
    lambda entry: entry.source[LATTICE_SURGERY].instruction.expect_time(1),
)

results.as_frame()

### Throttling the Algorithm

We can slow down each lattice-surgery step to conserve qubit count. QRE
examines each slow-down factor and reports Pareto-optimal configurations.

In [ ]:
new_trace_query = (
    app_query
    * PSSPC.q(num_ts_per_rotation=[15, 16, 17, 18])
    * LatticeSurgery.q(
        slow_down_factor=[1.0 * j for j in range(2, 31)]
    )
)

new_results = estimate(
    app, arch, isa_query, new_trace_query,
    max_error=0.01,
    name="new Majorana e-5, 3-aux",
)
new_results.add_column(
    "compute_distance",
    lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE],
)
new_results.add_column(
    "compute qubits",
    lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS],
)
new_results.add_column(
    "num_ts_per_rotation",
    lambda entry: entry.properties[NUM_TS_PER_ROTATION],
)
new_results.add_factory_summary_column()

new_results.as_frame()

## Tradeoff Curves

We use matplotlib to visualize the space–time tradeoff.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

combined = pd.concat([results.as_frame(), new_results.as_frame()])
plt.figure(figsize=(10, 6))
for name, group in combined.groupby("name"):
    plt.scatter(group["qubits"], group["runtime"],
               marker="x", label=name)
plt.xlabel("Total Qubits")
plt.ylabel("Runtime (seconds)")
plt.legend()
plt.show()